# Predict sql injection attacks

### Initail package imports

In [1]:
import pandas as pd
import tensorflow.keras
import os
import re
import nltk
#nltk.download('stopwords')
from nltk.corpus import stopwords 

### Data Cleaning Functions 

In [2]:
def cleanSqliData(data):
    for i in range(len(data)):
        data[i] = data[i].replace('\n', '')
        data[i] = data[i].replace('%20', ' ')
        data[i] = data[i].replace('=', ' = ')
        data[i] = data[i].replace('((', ' (( ')
        data[i] = data[i].replace('))', ' )) ')
        data[i] = data[i].replace('(', ' ( ')
        data[i] = data[i].replace(')', ' ) ')
        data[i] = data[i].lower()
    return data

In [3]:
stopWords = set(stopwords.words('english')) 
def removeStopWords(posts):
    filtered = ''
    for x in posts.split(' '):
        if x not in stopWords:
            filtered += ' ' + x
    return filtered

### Data Loading and Preprocessing

> SQLI Data from external sources

        Source: https://github.com/foospidy/payloads/tree/master/other/sqli
        File name: sqlifuzzer.txt
        Status : Ok
        Concerns:  
            1. Eliminated \n from each line ending
            2. UrlDecoding to handle space and other characters

In [4]:
path = './data/sqlifuzzer.txt'

In [5]:
sqlFuzzingLines = []
file = open(path, "r")
for x in file:
    sqlFuzzingLines.append(x)

In [6]:
sqlFuzzingLines = cleanSqliData(sqlFuzzingLines) 

        Source: https://github.com/foospidy/payloads/tree/master/other/sqli
        File Name: camoufl4g3.txt
        Status : ok


In [7]:
path = './data/camoufl4g3.txt'

In [8]:
sqlCamoufl4g3Lines = []
file = open(path, "r")
for x in file:
    sqlCamoufl4g3Lines.append(x)

In [9]:
sqlCamoufl4g3Lines = cleanSqliData(sqlCamoufl4g3Lines)

        Source: https://github.com/foospidy/payloads/tree/master/other/sqli
        File Name: libinjection-bypasses.txt
        Status: Ok
        Concerns: Strip &()o1: from start

In [10]:
path = './data/libinjection-bypasses.txt'

In [11]:
sqlBypassLines=[]
file = open(path, "r")
for x in file:
    sqlBypassLines.append(x)

In [12]:
sqlBypassLines = cleanSqliData(sqlBypassLines)

In [13]:
# if don't want &(*)* sign in beginning of each sentence then run next code else don't

for i in range(len(sqlBypassLines)):
    sentence = sqlBypassLines[i]
    try:
        sqlBypassLines[i] = sentence.split(':')[1]
    except:
        pass
    

        Source: https://github.com/foospidy/payloads/tree/master/other/sqli
        File Name: sqli_OWASP.txt
        Status: ok

In [14]:
path = './data/sqli_OSWAP.txt'

In [15]:
sqlOwaspLines = []
file = open(path, "r")
for x in file:
    sqlOwaspLines.append(x)

In [16]:
sqlOwaspLines = cleanSqliData(sqlOwaspLines)

        Source: https://github.com/danielmiessler/SecLists/tree/master/Fuzzing/SQLi
        File Name: Generic-SQLi.txt
        Status: Ok
        Concerns: UrlDecoding to handle space and other characters

In [17]:
path = './data/Generic-SQLi.txt'

In [18]:
genericSqliLines = []
file = open(path, "r")
for x in file:
    genericSqliLines.append(x)

In [19]:
genericSqliLines = cleanSqliData(genericSqliLines)

        Source: https://github.com/mrsuman2002/SQL-Injection-Authentication-Bypass-Cheat-Sheet
        File Name: SQL_Injection_Cheat_Sheet.txt
        Status: Ok

In [20]:
path = './data/SQL_Injection_Cheat_Sheet.txt'

In [21]:
sqlInjectionCheatSheetLines = []
file = open(path, "r")
for x in file:
    sqlInjectionCheatSheetLines.append(x)

In [22]:
sqlInjectionCheatSheetLines = cleanSqliData(sqlInjectionCheatSheetLines)

> Plain Data from external sources

        Source: https://github.com/tungpv98/Detect-Sql-Injection-by-Machine-Learning/tree/ebeb3287931677a3a42f11a7a08dbc13e374ff05/Sql-Injection/source/trainingdata
        File name: plain.txt
        Concerns: 
            1. Keep seperator as Aw3s0meSc0t7
            2. Remove last few records which containing URLs

In [23]:
path = './data/'
file = "plain.txt"

In [24]:
df = pd.read_csv(os.path.join(path,file), sep='Aw3s0meSc0t7', names=['benign'], header=None, engine='python')

In [25]:
plain_text = df['benign'].values  # get sentences

In [26]:
# convert from list to string
plain_data = ''
for x in plain_text:
    plain_data += " " + x

In [27]:
plain_data = removeStopWords(plain_data)  # remove stop words
plain_data = plain_data.split('.')        # split sentences

In [28]:
# seperate words inside tags
for i in range(len(plain_data)):
    plain_data[i] = plain_data[i].replace('<', ' <')
    plain_data[i] = plain_data[i].replace('>', '> ')
    plain_data[i] = plain_data[i].replace('=', ' = ')

In [29]:
print("Benign records: ", len(plain_data))

Benign records:  5369


> Plain Data (Self Created)

In [30]:
# read self created benign data

path = './data/benign_data.txt'
benign_data = []
file = open(path, "r")
for x in file:
    benign_data.append(x)


In [31]:
len(benign_data)

620

In [32]:
benign_sentence = []
for i in benign_data:
    sentences = i.split('.')
    
    for sentence in sentences:
        benign_sentence.append(sentence)

In [33]:
len(benign_sentence)

1167

> SQLi Data (Self Created)

In [34]:
# read self created sqli data

path = './data/sqli_for_training.txt'
sqli_data = []
file = open(path, "r")
for x in file:
    sqli_data.append(x)


In [35]:
len(sqli_data)

330

### Statistics

In [36]:
print(f'''
    SQLi:
    \tFuzzing :\t\t\t{len(sqlFuzzingLines)}
    \tCamoufl4gs :\t\t\t{len(sqlCamoufl4g3Lines)}
    \tBypass :\t\t\t{len(sqlBypassLines)}
    \tOwasp :\t\t\t\t{len(sqlOwaspLines)}
    \tGeneric :\t\t\t{len(genericSqliLines)}
    \tSql Injection Cheat Sheet:\t{len(sqlInjectionCheatSheetLines)}
    \tSQLi(Self) :\t\t\t{len(sqli_data)}
    =========================================================
    \t\tTotal SQLi data :\t{ len(genericSqliLines) + len(sqlOwaspLines) + len(sqlBypassLines) + len(sqlCamoufl4g3Lines) + len(sqlFuzzingLines) + len(sqlInjectionCheatSheetLines) + len(sqli_data)} 
    Plain/Benign:
    \tPlain(External) :\t\t{len(plain_data)}
    \tBenign(Self) :\t\t\t{len(benign_sentence)}
    =========================================================
    \t\tTotal Plain Data :\t{ len(benign_sentence) + len(plain_data) }
    ---------------------------------------------------------
    \t\t\tTotal Records :\t{ len(genericSqliLines) + len(sqlOwaspLines) + len(sqlBypassLines) + len(sqlCamoufl4g3Lines) + len(sqlFuzzingLines) + len(sqlInjectionCheatSheetLines) + len(sqli_data) + len(benign_sentence) + len(plain_data)}
    ''')


    SQLi:
    	Fuzzing :			86
    	Camoufl4gs :			77
    	Bypass :			474
    	Owasp :				6692
    	Generic :			308
    	Sql Injection Cheat Sheet:	47
    	SQLi(Self) :			330
    		Total SQLi data :	8014 
    Plain/Benign:
    	Plain(External) :		5369
    	Benign(Self) :			1167
    		Total Plain Data :	6536
    ---------------------------------------------------------
    			Total Records :	14550
    


### Collating all SQLi attack data

In [37]:
all_sqli_sentence = sqlOwaspLines + sqlBypassLines + sqlCamoufl4g3Lines + sqlFuzzingLines + genericSqliLines + sqlInjectionCheatSheetLines + sqli_data

In [38]:
len(all_sqli_sentence)

8014

### Collating all Plain/Benign data

In [39]:
all_plain_sentence = benign_sentence + plain_data

In [40]:
len(all_plain_sentence)

6536

### Final Cleaning of data before Generating Combined Data

>>> Converting numerics in sqli data to standard string

In [41]:
# replace numeric values by a standard string 'numeric'
def optional_numeric_to_numeric(all_sqli_sentence):
    for i in range(len(all_sqli_sentence)):
        all_sqli_sentence[i]=re.sub(r"\d+", "numeric",all_sqli_sentence[i])    
    return all_sqli_sentence

In [42]:
all_sqli_sentence = optional_numeric_to_numeric(all_sqli_sentence)

In [43]:
all_sqli_sentence[:25]

['\' or "',
 '-- or # ',
 "' or 'numeric",
 "' or numeric -- -",
 '" or ""  =  "',
 '" or numeric  =  numeric -- -',
 "' or ''  =  '",
 "' = '",
 "'like'",
 "' = numeric--+",
 ' or numeric = numeric',
 "' or 'x' = 'x",
 "' and id is null; --",
 "'''''''''''''union select 'numeric",
 'and numeric',
 'and numeric',
 'and true',
 'and false',
 'numeric-false',
 'numeric-true',
 'numeric*numeric',
 '-numeric',
 "numeric' order by numeric--+",
 "numeric' order by numeric--+",
 "numeric' order by numeric--+"]

>>> Removing Empty or single character strings from benign data

In [44]:
def optional_single_char_string_removal(all_plain_sentence):
    for i in range(len(all_plain_sentence)-1,-1,-1):
        if all_plain_sentence[i] == ' ' or all_plain_sentence[i] == '' or all_plain_sentence[i] == '\n' or all_plain_sentence[i] == '\t' or all_plain_sentence[i] == '\r':
            all_plain_sentence.remove(all_plain_sentence[i])
    return all_plain_sentence

In [45]:
all_plain_sentence = optional_single_char_string_removal(all_plain_sentence)

In [46]:
all_plain_sentence[:5]

['Remember to give benign data',
 ' eat bread or butter',
 ' can you wait for me',
 'wait for me',
 ' waitfor me']

## Post Data Cleaning Statistics

In [47]:
print(f'''
#####################################
####        Post Cleaning :      ####
#####################################
Total SQLi Data:\t{len(all_sqli_sentence)}
Total Plain Data:\t{len(all_plain_sentence)}
=====================================
Combined Total Data:\t{len(all_sqli_sentence)+len(all_plain_sentence)}''')


#####################################
####        Post Cleaning :      ####
#####################################
Total SQLi Data:	8014
Total Plain Data:	6275
Combined Total Data:	14289


### Generating single merged data file including both type of data, Benign and SQLi (with lables)

In [48]:
combined_data = []

In [49]:
# Setting SQLi data lable as 1

for i in all_sqli_sentence:
    combined_data.append((i,1))

In [50]:
# Setting Benign data lable as 0

for i in all_plain_sentence:
    combined_data.append((i,0))

In [51]:
# convert to dataframe
df=pd.DataFrame(combined_data,columns=['Sentence','Label'])

In [52]:
df.head()

,Sentence,Label
0,"' or """,1
1,-- or #,1
2,' or 'numeric,1
3,' or numeric -- -,1
4,""" or """" = """,1


### Save data as csv

In [53]:
df.to_csv('sqli.csv', index=False, encoding='utf-16')

### Reading Data from CSV

In [54]:
df=pd.read_csv('sqli.csv',encoding='utf-16')
df.drop_duplicates(inplace= True, ignore_index=True)

In [55]:
df.head()

,Sentence,Label
0,"' or """,1
1,-- or #,1
2,' or 'numeric,1
3,' or numeric -- -,1
4,""" or """" = """,1


In [56]:
len(df[df['Label']==0])

5879

In [57]:
len(df[df['Label']==1])

5360

In [58]:
#Vectorization NEW METHOD
def lNewStringVectorizer(inp=''): # can take df or string
    import pandas as pd
    import re
    
    def Newvectorizer(df):
        def num_of_SQLI_keywords(df):###############################SQL keyword count##########################
            SQL=["ALL","ADD CONSTRAINT","ALTER","DROP","ALTER COLUMN","ALTER TABLE","AND","ANY","AS","ASC","BACKUP DATABASE","BETWEEN","CASE","CHECK","COLUMN","CONSTRAINT","OPENROWSET","CREATE","INDEX","OR REPLACE VIEW","TABLE","PROCEDURE","UNIQUE INDEX","VIEW","DATABASE","DEFAULT","DELETE","DESC","DECLARE","DISTINCT","DROP COLUMN","DROP CONSTRAINT","DROP DATABASE","DROP DEFAULT","DROP INDEX","DROP TABLE","DROP VIEW","EXEC","EXISTS","FOREIGN KEY","FROM","FULL OUTER JOIN","GROUP BY","HAVING","IN","INDEX","INNER JOIN","INSERT INTO","INSERT INTO SELECT","SELECT USER","IS NULL","IS NOT NULL","LEFT JOIN","LIKE","LIMIT","NOT","NOT NULL","OR","ORDER BY","OUTER JOIN","PRIMARY KEY","PROCEDURE","RIGHT JOIN","ROWNUM","SELECT","DISTINCT","OUTFILE","INTO OUTFILE","SET","TOP","TRUNCATE TABLE","UNION","UNIQUE","UPDATE","VALUES","VIEW","WHERE","LOAD DATA","INFILE","CONCAT", "LOAD_FILE"]
            def countkey(k,r):
                symbols=['~','`','!','@','#','$','%','^','&','*','(',')','-','_','+','=','{','}','[',']','|','\\','/',':',';','"',"'",'<','>',',','?','.']
                rr=r['Sentence']
                for j in symbols:
                    rr=rr.replace('  ',' ')
                    rr=rr.replace(j,' '+j+' ')
                    rr=rr.replace('  ',' ')
                rr=" "+rr+" "
                rr=rr.lower()
                return rr.count(' '+k.lower()+' ') + rr.count(' '+k+' ')
            for a in SQL:
                df[a]=df.apply (lambda row: countkey(a,row), axis=1)
            return df

        def num_of_unsual_comb(df):###############################unusual combo count##########################
            combi_regexes={
                    "Number'":r"\d+'",
                    "Number":r"numeric",
                    "+Number,":r"\+\d+,",
                    "@@":r"@@[a-zA-Z]",
                    "Number--":r"\d+--",
                    "Alphabets--":r"[a-zA-Z]+--",
                    '%NumberAlphabet':r"%\d+[a-zA-Z]*",
                    '0x2f Encoded':r"0x\d+([a-zA-Z]|)",
                    'Multiline Comments':r"(\\/*)|(\*\/)",
                    'Multiline Comments with MYSQL syntax': r"\/\*! *\d+",
                    'Always true and other such conditions': r"(\"|')(( *\))|) ((or)|(OR)) ((\d+(=|>|<|!=|<>)\d+)|([a-zA-Z]+=[a-zA-Z]+))(( )|)(((--)|(\/\*)|(#))|)",
            }
            
            def countcombo(k,r):
                return len(re.findall(k, r['Sentence']))

            for a in combi_regexes:
                df[a]=df.apply (lambda row: countcombo(combi_regexes[a],row), axis=1)

            return df

        def num_of_symbols(df):
              symbols=['~','`','!','@','#','$','%','^','&','*','(',')','-','_','+','=','{','}','[',']','|','\\','/',':',';','"',"'",'<','>',',','❡','?']

              def countsymbols(symbol,row):
                  string = row['Sentence']
                  string = " " + string + " "
                  string = string.lower()
                  return  string.count(symbol)

              for symbol in symbols:
                  df[symbol] = df.apply (lambda row: countsymbols(symbol, row), axis=1)
              return df

        #calculate length
        df['rlen'] = df['Sentence'].str.split(" ").str.join("").str.len()
        df.dropna(inplace=True) 
        #count keywords
        df = num_of_SQLI_keywords(df)
        #count Unusual combos
        df = num_of_unsual_comb(df)
        #number of Symbols
        df = num_of_symbols(df)

        return df

    if(isinstance(inp, str)): #
        qdf = pd.DataFrame({'Sentence':[inp]})
        qdf=Newvectorizer(qdf)
        qdf=qdf[qdf.columns[1:]]
        return qdf
    else:
        return Newvectorizer(inp)

In [59]:
df=lNewStringVectorizer(inp=df)
df.head()

,Sentence,Label,rlen,ALL,ADD CONSTRAINT,ALTER,DROP,ALTER COLUMN,ALTER TABLE,AND,...,/,:,;,"""",',<,>,",",❡,?
0,"' or """,1,4,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,0,0,0
1,-- or #,1,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,' or 'numeric,1,11,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,0
3,' or numeric -- -,1,13,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,""" or """" = """,1,7,0,0,0,0,0,0,0,...,0,0,0,4,0,0,0,0,0,0


### New Vectorizer

In [94]:
#Vectorization NEW METHOD
def NewStringVectorizer(inp=''): # can take df or string
    import pandas as pd
    import re
    
    def Newvectorizer(df):
        def num_of_SQLI_keywords(df):###############################SQL keyword count##########################
            SQL=["ALL","ADD CONSTRAINT","ALTER","DROP","ALTER COLUMN","ALTER TABLE","AND","ANY","AS","ASC","BACKUP DATABASE","BETWEEN","CASE","CHECK","COLUMN","CONSTRAINT","OPENROWSET","CREATE","INDEX","OR REPLACE VIEW","TABLE","PROCEDURE","UNIQUE INDEX","VIEW","DATABASE","DEFAULT","DELETE","DESC","DECLARE","DISTINCT","DROP COLUMN","DROP CONSTRAINT","DROP DATABASE","DROP DEFAULT","DROP INDEX","DROP TABLE","DROP VIEW","EXEC","EXISTS","FOREIGN KEY","FROM","FULL OUTER JOIN","GROUP BY","HAVING","IN","INDEX","INNER JOIN","INSERT INTO","INSERT INTO SELECT","SELECT USER","IS NULL","IS NOT NULL","LEFT JOIN","LIKE","LIMIT","NOT","NOT NULL","OR","ORDER BY","OUTER JOIN","PRIMARY KEY","PROCEDURE","RIGHT JOIN","ROWNUM","SELECT","DISTINCT","OUTFILE","INTO OUTFILE","SET","TOP","TRUNCATE TABLE","UNION","UNIQUE","UPDATE","VALUES","VIEW","WHERE","LOAD DATA","INFILE","CONCAT", "LOAD_FILE"]
            def countkey(k,r):
                symbols=['~','`','!','@','#','$','%','^','&','*','(',')','-','_','+','=','{','}','[',']','|','\\','/',':',';','"',"'",'<','>',',','?','.']
                rr=r['Sentence']
                for j in symbols:
                    rr=rr.replace('  ',' ')
                    rr=rr.replace(j,' '+j+' ')
                    rr=rr.replace('  ',' ')
                rr=" "+rr+" "
                rr=rr.lower()
                return rr.count(' '+k.lower()+' ') + rr.count(' '+k+' ')
            for a in SQL:
                df[a]=df.apply (lambda row: countkey(a,row), axis=1)
            return df

        def num_of_unsual_comb(df):###############################unusual combo count##########################
            combi_regexes={
                    "Number'":r"\d+'",
                    "Number":r"numeric",
                    "+Number,":r"\+\d+,",
                    "@@":r"@@[a-zA-Z]",
                    "Number--":r"\d+--",
                    "Alphabets--":r"[a-zA-Z]+--",
                    '%NumberAlphabet':r"%\d+[a-zA-Z]*",
                    '0x2f Encoded':r"0x\d+([a-zA-Z]|)",
                    'Multiline Comments':r"(\\/*)|(\*\/)",
                    'Multiline Comments with MYSQL syntax': r"\/\*! *\d+",
                    'Always true and other such conditions': r"(\"|')(( *\))|) ((or)|(OR)) ((\d+(=|>|<|!=|<>)\d+)|([a-zA-Z]+=[a-zA-Z]+))(( )|)(((--)|(\/\*)|(#))|)",
            }
            
            def countcombo(k,r):
                return len(re.findall(k, r['Sentence']))

            for a in combi_regexes:
                df[a]=df.apply (lambda row: countcombo(combi_regexes[a],row), axis=1)

            return df

        def num_of_symbols(df):
              symbols=['~','`','!','@','#','$','%','^','&','*','(',')','-','_','+','=','{','}','[',']','|','\\','/',':',';','"',"'",'<','>',',','❡','?']

              def countsymbols(symbol,row):
                  string = row['Sentence']
                  string = " " + string + " "
                  string = string.lower()
                  return  string.count(symbol)

              for symbol in symbols:
                  df[symbol] = df.apply (lambda row: countsymbols(symbol, row), axis=1)
              return df

        #calculate length
        df['rlen'] = df['Sentence'].str.split(" ").str.join("").str.len()
        df.dropna(inplace=True) 
        #count keywords
        df = num_of_SQLI_keywords(df)
        #count Unusual combos
        df = num_of_unsual_comb(df)
        #number of Symbols
        df = num_of_symbols(df)

        return df

    if(isinstance(inp, str)): #
        qdf = pd.DataFrame({'Sentence':[inp]})
        qdf=Newvectorizer(qdf)
        qdf=qdf[qdf.columns[1:]]
        return qdf
    else:
        return Newvectorizer(inp)

nv=NewStringVectorizer


### Suffling data and spilting

In [95]:
df = df.sample(frac=1).reset_index(drop=True)
df.head()

,Sentence,Label,rlen,ALL,ADD CONSTRAINT,ALTER,DROP,ALTER COLUMN,ALTER TABLE,AND,...,/,:,;,"""",',<,>,",",❡,?
0,numeric' and ( numeric = numeric ) *numeric--...,1,42,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
1,numeric and ( select ( case when ( numeric ...,1,113,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
2,what is title of email\n,0,19,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,and numeric = numeric and 'pklz' = 'pkly,1,33,0,0,0,0,0,0,2,...,0,0,0,0,3,0,0,0,0,0
4,-numeric ) ) or numeric = numeric\unumeric,1,36,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [96]:
#Drop Nan Values
df = df.dropna(subset=['Sentence'])
df = df.sample(frac=1).reset_index(drop=True)

In [63]:
x = df[df.columns[2:]]

In [64]:
x.shape

(11239, 121)

In [65]:
y = df['Label']

In [66]:
from sklearn.model_selection import train_test_split

In [67]:
# split train test data

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [68]:
print(f'''x_train.shape :\t{x_train.shape}
y_train.shape :\t{ y_train.shape }
x_test.shape :\t{ x_test.shape }
y_test.shape :\t{ y_test.shape }
''')

x_train.shape :	(8991, 121)
y_train.shape :	(8991,)
x_test.shape :	(2248, 121)
y_test.shape :	(2248,)



### Model Prepration Imports

In [69]:
%pip install --upgrade wandb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [70]:
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorboard.plugins.hparams import api as hp
from sklearn.metrics import precision_score, recall_score, accuracy_score
import wandb
from wandb.keras import WandbCallback

### Confusion Matrix

In [71]:
def calculateAccuracy(truePositive,trueNegative,falsePositive,falseNegative):
    return ( truePositive + trueNegative ) / ( truePositive + trueNegative + falsePositive + falseNegative )

def calculatePrecision(truePositive,falsePositive):
    return truePositive / ( truePositive + falsePositive )

def calculateRecall(truePositive,falseNegative):
    return truePositive / (truePositive+falseNegative)

def generateConfusionMatrix(test,prediction):
    truePositive = 0
    trueNegative = 0
    falsePositive = 0
    falseNegative = 0

    for true, pred in zip(test,prediction):
        if true == 1:
            if pred == true:
                truePositive += 1
            else:
                falseNegative += 1
        elif true == 0:
            if pred == true:
                trueNegative += 1
            else:
                falsePositive += 1
    print(f'''True Positive:\t{truePositive}
    True Negative:\n{trueNegative}
    False Positive:\n{falsePositive}
    False Negative:\n{falseNegative}
    ''')
    accuracy = calculateAccuracy(truePositive, trueNegative, falsePositive, falseNegative)
    precision = calculatePrecision(truePositive, falsePositive)
    recall = calculateRecall(truePositive, falseNegative)
    
    return (accuracy, precision, recall)

### Model Prepration and training

In [72]:
losses=['binary_crossentropy','categorical_crossentropy','kl_divergence']
optimizers=['adam','sgd','rmsprop','adagrad','adadelta','adamax']
models={}
for a in losses:
   for b in optimizers:
       print("\n---------------------------------------------------------------\n\n############################ Model: "+a+"@"+b+" ############################\n")
       tensorflow.keras.backend.clear_session()
       #Weight And Biases Login
       #run=wandb.init(project="iwaf",name="NN Model# "+a+"@"+b) # uncomment this line to run on wandb
       input_dim = x_train.shape[1]  # Number of features
       model = Sequential()
       model.add(tensorflow.keras.layers.BatchNormalization(input_dim=input_dim))
       model.add(tensorflow.keras.layers.Dense(1024, activation='relu'))
       model.add(tensorflow.keras.layers.Dense(1024, activation='relu'))
       #model.add(tensorflow.keras.layers.Dense(512, activation='relu'))
       #model.add(tensorflow.keras.layers.Dense(128, activation='relu'))
       # model.add(keras.layers.Dense(20,  activation='relu'))
       # model.add(keras.layers.Dense(10,  activation='tanh'))
       # model.add(layers.Flatten())
       # model.add(keras.layers.Dense(1024, activation='relu'))
       #model.add(tensorflow.keras.layers.BatchNormalization())
       model.add(tensorflow.keras.layers.Dropout(0.5))
       model.add(tensorflow.keras.layers.Dense(1, activation='sigmoid'))
       model.compile(loss = a, optimizer = b, metrics = ['accuracy'])
       model.summary()
       print()
       model.fit(x_train,y_train,
                           epochs=20,
                           verbose=True,
                           validation_data=(x_test, y_test),
                           batch_size=1024,
                           #callbacks=[WandbCallback()], # uncomment this line to use wandb
                           )
       models[a+"@"+b]=model
       del model
       tensorflow.keras.backend.clear_session()
       #run.finish() # uncomment this line to use wandb


---------------------------------------------------------------

############################ Model: binary_crossentropy@adam ############################

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
batch_normalization (BatchNo (None, 121)               484       
_________________________________________________________________
dense (Dense)                (None, 1024)              124928    
_________________________________________________________________
dense_1 (Dense)              (None, 1024)              1049600   
_________________________________________________________________
dropout (Dropout)            (None, 1024)              0         
_________________________________________________________________
dense_2 (Dense)              (None, 1)                 1025      
Total params: 1,176,037
Trainable params: 1,175,795
Non-trainable params: 242
___________________

9/9 [==============================] - 2s 238ms/step - loss: 0.2707 - accuracy: 0.8814 - val_loss: 0.3109 - val_accuracy: 0.8968
Epoch 2/20
9/9 [==============================] - 2s 187ms/step - loss: 0.0878 - accuracy: 0.9748 - val_loss: 0.2839 - val_accuracy: 0.9284
Epoch 3/20
9/9 [==============================] - 1s 165ms/step - loss: 0.0720 - accuracy: 0.9782 - val_loss: 0.1815 - val_accuracy: 0.9773
Epoch 4/20
9/9 [==============================] - 1s 163ms/step - loss: 0.0623 - accuracy: 0.9806 - val_loss: 0.2358 - val_accuracy: 0.9453
Epoch 5/20
9/9 [==============================] - 1s 147ms/step - loss: 0.0635 - accuracy: 0.9814 - val_loss: 0.1382 - val_accuracy: 0.9831
Epoch 6/20
9/9 [==============================] - 2s 194ms/step - loss: 0.0458 - accuracy: 0.9871 - val_loss: 0.0980 - val_accuracy: 0.9831
Epoch 7/20
9/9 [==============================] - 2s 170ms/step - loss: 0.0534 - accuracy: 0.9867 - val_loss: 0.0831 - val_accuracy: 0.9818
Epoch 8/20
9/9 [===============

9/9 [==============================] - 1s 147ms/step - loss: 0.7157 - accuracy: 0.5801 - val_loss: 0.7194 - val_accuracy: 0.5409
Epoch 3/20
9/9 [==============================] - 2s 175ms/step - loss: 0.7086 - accuracy: 0.6006 - val_loss: 0.7105 - val_accuracy: 0.5698
Epoch 4/20
9/9 [==============================] - 2s 172ms/step - loss: 0.7053 - accuracy: 0.6099 - val_loss: 0.7039 - val_accuracy: 0.6045
Epoch 5/20
9/9 [==============================] - 2s 214ms/step - loss: 0.6994 - accuracy: 0.6233 - val_loss: 0.6985 - val_accuracy: 0.6335
Epoch 6/20
9/9 [==============================] - 2s 194ms/step - loss: 0.6984 - accuracy: 0.6365 - val_loss: 0.6936 - val_accuracy: 0.6619
Epoch 7/20
9/9 [==============================] - 2s 171ms/step - loss: 0.6915 - accuracy: 0.6481 - val_loss: 0.6892 - val_accuracy: 0.6931
Epoch 8/20
9/9 [==============================] - 2s 188ms/step - loss: 0.6865 - accuracy: 0.6658 - val_loss: 0.6850 - val_accuracy: 0.7144
Epoch 9/20
9/9 [===============

9/9 [==============================] - 2s 176ms/step - loss: 5.6495e-08 - accuracy: 0.5856 - val_loss: 5.8279e-08 - val_accuracy: 0.7611
Epoch 4/20
9/9 [==============================] - 1s 144ms/step - loss: 5.6495e-08 - accuracy: 0.5849 - val_loss: 5.8279e-08 - val_accuracy: 0.7375
Epoch 5/20
9/9 [==============================] - 1s 139ms/step - loss: 5.6495e-08 - accuracy: 0.5794 - val_loss: 5.8279e-08 - val_accuracy: 0.6993
Epoch 6/20
9/9 [==============================] - 2s 169ms/step - loss: 5.6495e-08 - accuracy: 0.5815 - val_loss: 5.8279e-08 - val_accuracy: 0.6432
Epoch 7/20
9/9 [==============================] - 1s 153ms/step - loss: 5.6495e-08 - accuracy: 0.5771 - val_loss: 5.8279e-08 - val_accuracy: 0.6174
Epoch 8/20
9/9 [==============================] - 1s 144ms/step - loss: 5.6495e-08 - accuracy: 0.5860 - val_loss: 5.8279e-08 - val_accuracy: 0.6005
Epoch 9/20
9/9 [==============================] - 1s 147ms/step - loss: 5.6495e-08 - accuracy: 0.5849 - val_loss: 5.8279e-0

9/9 [==============================] - 1s 159ms/step - loss: 5.6495e-08 - accuracy: 0.5679 - val_loss: 5.8279e-08 - val_accuracy: 0.5351
Epoch 3/20
9/9 [==============================] - 2s 201ms/step - loss: 5.6495e-08 - accuracy: 0.5781 - val_loss: 5.8279e-08 - val_accuracy: 0.5329
Epoch 4/20
9/9 [==============================] - 1s 141ms/step - loss: 5.6495e-08 - accuracy: 0.5781 - val_loss: 5.8279e-08 - val_accuracy: 0.5338
Epoch 5/20
9/9 [==============================] - 1s 160ms/step - loss: 5.6495e-08 - accuracy: 0.5762 - val_loss: 5.8279e-08 - val_accuracy: 0.5343
Epoch 6/20
9/9 [==============================] - 1s 143ms/step - loss: 5.6495e-08 - accuracy: 0.5631 - val_loss: 5.8279e-08 - val_accuracy: 0.5356
Epoch 7/20
9/9 [==============================] - 1s 153ms/step - loss: 5.6495e-08 - accuracy: 0.5741 - val_loss: 5.8279e-08 - val_accuracy: 0.5351
Epoch 8/20
9/9 [==============================] - 1s 146ms/step - loss: 5.6495e-08 - accuracy: 0.5706 - val_loss: 5.8279e-0

9/9 [==============================] - 1s 167ms/step - loss: 5.6495e-08 - accuracy: 0.3669 - val_loss: 5.8279e-08 - val_accuracy: 0.3003
Epoch 2/20
9/9 [==============================] - 1s 152ms/step - loss: 5.6495e-08 - accuracy: 0.3734 - val_loss: 5.8279e-08 - val_accuracy: 0.2931
Epoch 3/20
9/9 [==============================] - 2s 174ms/step - loss: 5.6495e-08 - accuracy: 0.3706 - val_loss: 5.8279e-08 - val_accuracy: 0.3003
Epoch 4/20
9/9 [==============================] - 1s 156ms/step - loss: 5.6495e-08 - accuracy: 0.3664 - val_loss: 5.8279e-08 - val_accuracy: 0.3043
Epoch 5/20
9/9 [==============================] - 1s 154ms/step - loss: 5.6495e-08 - accuracy: 0.3659 - val_loss: 5.8279e-08 - val_accuracy: 0.3136
Epoch 6/20
9/9 [==============================] - 1s 146ms/step - loss: 5.6495e-08 - accuracy: 0.3733 - val_loss: 5.8279e-08 - val_accuracy: 0.3283
Epoch 7/20
9/9 [==============================] - 1s 152ms/step - loss: 5.6495e-08 - accuracy: 0.3748 - val_loss: 5.8279e-0

Epoch 1/20
9/9 [==============================] - 2s 167ms/step - loss: 0.0420 - accuracy: 0.4577 - val_loss: 0.0019 - val_accuracy: 0.4889
Epoch 2/20
9/9 [==============================] - 1s 148ms/step - loss: 3.8300e-05 - accuracy: 0.4739 - val_loss: 6.4990e-04 - val_accuracy: 0.4889
Epoch 3/20
9/9 [==============================] - 1s 147ms/step - loss: 3.4658e-06 - accuracy: 0.4739 - val_loss: 3.5674e-04 - val_accuracy: 0.4889
Epoch 4/20
9/9 [==============================] - 1s 148ms/step - loss: 6.2572e-07 - accuracy: 0.4739 - val_loss: 2.3690e-04 - val_accuracy: 0.4889
Epoch 5/20
9/9 [==============================] - 1s 129ms/step - loss: 1.6939e-07 - accuracy: 0.4739 - val_loss: 1.7659e-04 - val_accuracy: 0.4889
Epoch 6/20
9/9 [==============================] - 1s 147ms/step - loss: -2.0242e-07 - accuracy: 0.4739 - val_loss: 1.3762e-04 - val_accuracy: 0.4889
Epoch 7/20
9/9 [==============================] - 1s 147ms/step - loss: -1.3199e-07 - accuracy: 0.4739 - val_loss: 1.10

9/9 [==============================] - 2s 193ms/step - loss: 0.0414 - accuracy: 0.4695 - val_loss: 0.0055 - val_accuracy: 0.4889
Epoch 2/20
9/9 [==============================] - 2s 173ms/step - loss: 3.4346e-04 - accuracy: 0.4739 - val_loss: 0.0044 - val_accuracy: 0.4889
Epoch 3/20
9/9 [==============================] - 1s 150ms/step - loss: 1.2422e-04 - accuracy: 0.4739 - val_loss: 0.0031 - val_accuracy: 0.4889
Epoch 4/20
9/9 [==============================] - 1s 148ms/step - loss: 5.0780e-05 - accuracy: 0.4739 - val_loss: 0.0021 - val_accuracy: 0.4889
Epoch 5/20
9/9 [==============================] - 1s 150ms/step - loss: 2.4842e-05 - accuracy: 0.4739 - val_loss: 0.0014 - val_accuracy: 0.4889
Epoch 6/20
9/9 [==============================] - 2s 168ms/step - loss: 1.3032e-05 - accuracy: 0.4739 - val_loss: 8.9569e-04 - val_accuracy: 0.4889
Epoch 7/20
9/9 [==============================] - 1s 154ms/step - loss: 6.8048e-06 - accuracy: 0.4739 - val_loss: 5.7686e-04 - val_accuracy: 0.4889

9/9 [==============================] - 2s 226ms/step - loss: 0.3477 - accuracy: 0.4037 - val_loss: 0.3485 - val_accuracy: 0.2647
Epoch 2/20
9/9 [==============================] - 1s 132ms/step - loss: 0.3448 - accuracy: 0.3915 - val_loss: 0.3445 - val_accuracy: 0.3092
Epoch 3/20
9/9 [==============================] - 1s 135ms/step - loss: 0.3387 - accuracy: 0.3971 - val_loss: 0.3409 - val_accuracy: 0.3092
Epoch 4/20
9/9 [==============================] - 2s 219ms/step - loss: 0.3331 - accuracy: 0.3998 - val_loss: 0.3373 - val_accuracy: 0.3012
Epoch 5/20
9/9 [==============================] - 2s 180ms/step - loss: 0.3294 - accuracy: 0.4013 - val_loss: 0.3338 - val_accuracy: 0.3238
Epoch 6/20
9/9 [==============================] - 2s 184ms/step - loss: 0.3210 - accuracy: 0.4160 - val_loss: 0.3304 - val_accuracy: 0.3403
Epoch 7/20
9/9 [==============================] - 2s 182ms/step - loss: 0.3163 - accuracy: 0.4183 - val_loss: 0.3270 - val_accuracy: 0.3550
Epoch 8/20
9/9 [===============

### Model Metrics for various Models 

In [73]:
md = {}
for model in models:
    pred = models[model].predict(x_test)
    for i in range(len(pred)):
        pred[i] = 1 if pred[i] > 0.5 else 0
    print("\n---------------------------------------------------------------------\nAccuracy Score of "+model,"As per TF",accuracy_score(y_test,pred))
    print("####### Using Confusin Matrix #######\n")
    accuracy, precision, recall = generateConfusionMatrix(y_test, pred)
    if md.get(model.split('@')[0]) is None:
        md[model.split('@')[0]] = { model : accuracy }
    else:
        md[model.split('@')[0]][ model ] = accuracy
    print(model+"*"+"\n\tAccuracy : {0} \n\tPrecision : {1} \n\tRecall : {2}".format(accuracy, precision, recall))


---------------------------------------------------------------------
Accuracy Score of binary_crossentropy@adam As per TF 0.9893238434163701
####### Using Confusin Matrix #######

True Positive:	1078
    True Negative:
1146
    False Positive:
3
    False Negative:
21
    
binary_crossentropy@adam*
	Accuracy : 0.9893238434163701 
	Precision : 0.9972247918593895 
	Recall : 0.9808917197452229

---------------------------------------------------------------------
Accuracy Score of binary_crossentropy@sgd As per TF 0.9653024911032029
####### Using Confusin Matrix #######

True Positive:	1028
    True Negative:
1142
    False Positive:
7
    False Negative:
71
    
binary_crossentropy@sgd*
	Accuracy : 0.9653024911032029 
	Precision : 0.9932367149758454 
	Recall : 0.935395814376706

---------------------------------------------------------------------
Accuracy Score of binary_crossentropy@rmsprop As per TF 0.9902135231316725
####### Using Confusin Matrix #######

True Positive:	1082
    Tr

### Top 3 Models of individual Loss Functions across 6 optimizers

In [74]:
for i in md:
  md[i] = { k: v for k, v in sorted(md[i].items(),reverse=True, key=lambda item: item[1]) }
  #print(md[i])
  print("\n------------------------------------------\nTop 3 Models and optimizers of",i)
  c = 0
  for a in md[i]:
    if c > 2 : break
    print("# Using "+a.split("@")[1]+" as optimizer\t\tAccuracy: "+str(md[i][a])+"\t(Model Name: "+a+")")
    c += 1


------------------------------------------
Top 3 Models and optimizers of binary_crossentropy
# Using rmsprop as optimizer		Accuracy: 0.9902135231316725	(Model Name: binary_crossentropy@rmsprop)
# Using adam as optimizer		Accuracy: 0.9893238434163701	(Model Name: binary_crossentropy@adam)
# Using adamax as optimizer		Accuracy: 0.9879893238434164	(Model Name: binary_crossentropy@adamax)

------------------------------------------
Top 3 Models and optimizers of categorical_crossentropy
# Using rmsprop as optimizer		Accuracy: 0.5435943060498221	(Model Name: categorical_crossentropy@rmsprop)
# Using adam as optimizer		Accuracy: 0.5204626334519573	(Model Name: categorical_crossentropy@adam)
# Using adagrad as optimizer		Accuracy: 0.4986654804270463	(Model Name: categorical_crossentropy@adagrad)

------------------------------------------
Top 3 Models and optimizers of kl_divergence
# Using adam as optimizer		Accuracy: 0.4888790035587189	(Model Name: kl_divergence@adam)
# Using sgd as optim

### Saving Model And Vectorizer


In [97]:
from tensorflow.keras.models import load_model
import cloudpickle

best_model = 'binary_crossentropy@adam'
print("###Saving Model###")
try:
    models[best_model].save(best_model+'-Final-SQLI-Model.h5')
    print("\t\tSaved Model Successfully!")
except:
    print("\t\t Failed to save Model!")
print('\n###Saving Vectorizer###')
try:
    #with open('Final-SQLI-Vectorizer', 'wb') as fin:
    with open('New-Final-SQLI-Vectorizer', 'wb') as vecortizer:
        #pickle.dump(vectorizer, fin)
        cloudpickle.dump(nv, vecortizer)
    print("\t\tSaved Vectorizer Successfully!")
except:
    print("\t\t Failed to save Vectorizer!")

###Saving Model###
		Saved Model Successfully!

###Saving Vectorizer###
		Saved Vectorizer Successfully!


### Load Model, Vectorizer and Manual Testing

     This cell is also in file user_Data_predict.py

In [ ]:
%pip install --upgrade cloudpickle
%pip freeze

### Testing 

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow.keras as keras
from tensorflow.keras.models import load_model
import cloudpickle
import pandas as pd
import re

models = { "binary_crossentropy@adam" : load_model('binary_crossentropy@adam-Final-SQLI-Model.h5') }
md = {}
for model in models:
    pred = models[model].predict(x_test)
    for i in range(len(pred)):
        pred[i] = 1 if pred[i] > 0.5 else 0
    print("\n---------------------------------------------------------------------\nAccuracy Score of "+model,"As per TF",accuracy_score(y_test,pred))
    print("####### Using Confusin Matrix #######\n")
    accuracy, precision, recall = generateConfusionMatrix(y_test, pred)
    if md.get(model.split('@')[0]) is None:
        md[model.split('@')[0]] = { model : accuracy }
    else:
        md[model.split('@')[0]][ model ] = accuracy
    print(model+"*"+"\n\tAccuracy : {0} \n\tPrecision : {1} \n\tRecall : {2}".format(accuracy, precision, recall))

class Intelligent:
    mymodel = load_model('binary_crossentropy@adam-Final-SQLI-Model.h5')
    myvectorizer = cloudpickle.load(open('New-Final-SQLI-Vectorizer', 'rb'))
    myvectorizer = staticmethod(myvectorizer)
    def predict_sqli_attack(self,input_val=0,verbose=False):
        def clean_data(inp):
            inp = inp.replace('\n', '')
            inp = inp.replace('%20', ' ')
            inp = inp.replace('/', ' ')
            inp = inp.replace('_', ' _ ')
            inp = inp.replace('?', ' ? ')
            inp = inp.replace('.', ' . ')
            inp = inp.replace('=', ' ')
            inp = re.sub(r"\d+", "numeric", inp) 

            return inp

        def out(s):
            if verbose:
                print(s)
        repeat = True
        beautify = ''
        for i in range(20):
            beautify += "="
        zip=False
        if input_val == 0:
            zip = True
            out(beautify+"\nEnter 0 anytime to exit!\n"+beautify) 
            input_val = input("Give me some data to work on : \n")
            out(beautify)

        if input_val == '0':
            repeat = False    
        
        input_val = clean_data(input_val)
        clr_str = input_val

        # Vectorization with New Vectorizer
        print("Cleaned Input:",input_val)
        input_val = self.myvectorizer(inp=input_val)

        result = self.mymodel.predict(input_val)
        out(beautify) 

        if repeat == True and zip == True :
            if result > 0.5:
                print(result,"ALERT :::: This can be SQL injection")
            elif result <= 0.5:
                print(result,"It seems to be safe") 
            out(beautify)
            self.predict_sqli_attack()
        else:
            #if result>0.5:print(clr_str)
            return(result)

zop = Intelligent()

In [ ]:
zop.predict_sqli_attack()

# Other Possible Vectorization criteria Ideas

In [ ]:
def distance_between_words(string, word1, word2):
     
    if word1 == word2 :
       return 0
    words = string.lower().split(" ")
    min_dist = len(words)+1
    for index in range(len(words)):
        if words[index] == word1:
            for search in range(len(words)):
                if words[search] == word2:
                    curr = abs(index - search) - 1
                    if curr < min_dist:
                       min_dist = curr

    return min_dist    

In [ ]:
path = './data/sqli_for_training.txt'
test_word1='select'
test_word2='from'
test_data = []
file = open(path, "r")
t = {}
for x in file:
    w=distance_between_words(x , test_word1, test_word2)
    if t.get(str(w)) is None:
        t[str(w)]=0
    else:
        t[str(w)]+=1
    test_data.append(w)

from matplotlib import pyplot as plt
k=list(t.keys())
k = sorted(k, key=lambda x: int(x), reverse=False)
ww = {i: t[i] for i in k}
plt.bar(range(len(ww)), list(ww.values()), align='center')
plt.xticks(range(len(ww)), list(ww.keys()))
plt.title("Distance between `"+test_word1+"` and `"+test_word2+"`")
plt.show()

In [ ]:
path = './data/sqli_for_training.txt'
test_word1='from'
test_word2='where'
test_data = []
file = open(path, "r")
t = {}
for x in file:
    w=distance_between_words(x , test_word1, test_word2)
    if t.get(str(w)) is None:
        t[str(w)]=0
    else:
        t[str(w)]+=1
    test_data.append(w)

from matplotlib import pyplot as plt
k=list(t.keys())
k = sorted(k, key=lambda x: int(x), reverse=False)
ww = {i: t[i] for i in k}
plt.bar(range(len(ww)), list(ww.values()), align='center')
plt.xticks(range(len(ww)), list(ww.keys()))
plt.title("Distance between `"+test_word1+"` and `"+test_word2+"`")
plt.show()